# Toy Supervised Fine-Tuning from Scratch

## 1. Introduction

This notebook implements a small supervised fine-tuning (SFT) pipeline with PyTorch and Hugging Face Transformers. The dataset is 60 one-step addition questions. The training code is the whole lesson: it prepares labels, runs a forward pass, computes next-token loss, backpropagates, and updates the pretrained weights with AdamW.

SFT, for a decoder-only language model, is continued next-token training on examples that pair an input with a desired output. Pretraining teaches the model to continue text from a huge mixed corpus. SFT keeps that same prediction problem and changes two things: which strings are used, and which token positions count in the loss. In this notebook the desired output is a short answer such as `Answer: 12`, and the loss is applied only to that answer, including the end-of-sequence token.

## What this notebook demonstrates

- How an instruction and a response become one token sequence.
- Why prompt tokens are visible to the model and still excluded from the loss.
- How padding, attention masks, and a `DataLoader` build a batch.
- What happens in one optimization step: forward pass, loss, backward pass, AdamW update.
- How to score a held-out set with teacher-forced loss and with generated text.
- How to compare the original checkpoint with the fine-tuned checkpoint under the same prompt and the same decoder.

## What this notebook is not

This is not a production trainer and not a benchmark project. The toy set is here so the tensors stay small enough to inspect. A drop in training loss means the model assigns higher probability to the training answers while it is shown the correct earlier tokens. It does not, by itself, mean the model became a better calculator or a better assistant. The comparison at the end is the check that matters, and on this particular task the base model was already strong.

## Pipeline

```text
Pretrained Model
       |
       v
Instruction-Response Dataset
       |
       v
Tokenization and Label Preparation
       |
       v
Batching
       |
       v
Forward Pass
       |
       v
Loss Calculation
       |
       v
Backpropagation
       |
       v
Optimizer Update
       |
       v
Fine-Tuned Model
       |
       v
Evaluation and Comparison
```


## 2. Environment and dependencies

| Library | Role in this notebook |
|---|---|
| PyTorch | Tensors, `DataLoader`, backpropagation, and AdamW |
| Transformers | Loads the causal language model and its tokenizer |
| Python `json` | Reads the instruction-response files |
| `pathlib` | Builds paths to the data and the local checkpoint |

The model is a local folder, `models/Qwen3-0.6B-Base`. This notebook does not download weights. Place that checkpoint in the folder before running the cells. Its license is separate from the MIT license on this tutorial code.

The recorded run quoted later used the package versions in this environment (`torch` 2.14.0 and `transformers` 5.17.0) and left the model on CPU. This notebook selects CUDA, then Apple MPS, then CPU. Rerunning on another device, or with a different shuffle order, can change the exact loss values. To stay close to the recorded run, force CPU where the device is chosen below.


In [ ]:
import json
from pathlib import Path

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# The notebook lives in notebooks/. Accept either that directory or the repo root
# as the process working directory.
cwd = Path.cwd().resolve()
if (cwd / "data" / "train.json").is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "train.json").is_file():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/train.json. Start Jupyter from the repository "
        "root, or from the notebooks/ directory."
    )

BASE_MODEL_DIR = PROJECT_ROOT / "models" / "Qwen3-0.6B-Base"
SFT_MODEL_DIR = PROJECT_ROOT / "models" / "Qwen3-0.6B-SFT"
TRAIN_PATH = PROJECT_ROOT / "data" / "train.json"
HELDOUT_PATH = PROJECT_ROOT / "data" / "heldout.json"

# These match the run that produced the numbers quoted later.
# They are settings for this toy, not a general recommendation.
LEARNING_RATE = 5e-5
BATCH_SIZE = 4
NUM_EPOCHS = 5
MAX_NEW_TOKENS = 32

if not (BASE_MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(
        f"Place the Qwen3-0.6B-Base checkpoint in {BASE_MODEL_DIR}. "
        "This notebook does not download model weights."
    )

print("Project root:", PROJECT_ROOT)
print("Base model:  ", BASE_MODEL_DIR)
print("torch", torch.__version__)


## 3. Dataset preparation

Each example is one question and one target string. The target already includes the literal prefix `Answer:`, because that prefix is part of the behaviour we supervise. The model is not asked to invent the format during training. It is shown the format on every row.

```json
[
  {
    "question": "What is 2 + 3?",
    "answer": "Answer: 5"
  }
]
```

`data/train.json` has 40 rows. Those rows are the only examples that enter the loss and update the weights. `data/heldout.json` has 20 rows. Every number pair in the held-out file is absent from training, so a training row cannot reappear with the operands swapped. Held-out rows are for measurement after training. If they were mixed into the training loader, the gradient would be fitted to the same items used to judge the model, and the evaluation would no longer be a check on unseen pairs.

The questions are intentionally easy. `Qwen3-0.6B-Base` can already answer many of them. That makes the later comparison informative: SFT can only be credited with a gain that the starting checkpoint did not already have.


In [ ]:
with TRAIN_PATH.open(encoding="utf-8") as f:
    train_data = json.load(f)

with HELDOUT_PATH.open(encoding="utf-8") as f:
    heldout_data = json.load(f)

print(f"Training examples: {len(train_data)}")
print(f"Held-out examples: {len(heldout_data)}")
print("First training row:", train_data[0])
print("First held-out row:", heldout_data[0])


## 4. Tokenization

The model does not read characters. A tokenizer maps a string to a list of integer ids, and those ids are the `input_ids` the model embeds. One word is not guaranteed to be one id. A number, a space, or the word `Question` may be one piece or several, depending on the vocabulary. The code below prints the pieces for a real row so the split is visible.

We build two strings and encode them separately:

```text
prompt = "Question: " + question + "\n"
answer = "Answer: <sum>" + eos
input_ids = encode(prompt) + encode(answer)
```

Encoding the two sides separately, with `add_special_tokens=False`, keeps a clean cut between context and target. Encoding the joined string and then guessing where the prompt ends is unsafe, because a byte-pair tokenizer can merge across that boundary.

The end-of-sequence token (`eos`) is appended to the answer string. It is a real supervised target. It teaches the model that the answer is finished, which is what allows generation to stop. Leaving it out often makes generations run on until `max_new_tokens`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)

# Qwen's tokenizer has no pad token. Reuse the end token as padding, and later
# keep those positions out of the loss with -100. Otherwise the model would be
# trained to emit an extra end token wherever a row was padded.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("eos token:", repr(tokenizer.eos_token), "id:", tokenizer.eos_token_id)
print("pad token:", repr(tokenizer.pad_token), "id:", tokenizer.pad_token_id)


def make_example(example):
    """Turn one question/answer record into ids, a mask, and labels."""
    prompt = "Question: " + example["question"] + "\n"
    answer = example["answer"] + tokenizer.eos_token

    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    answer_ids = tokenizer.encode(answer, add_special_tokens=False)
    input_ids = prompt_ids + answer_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }


train_rows = [make_example(example) for example in train_data]

first = train_data[0]
first_row = train_rows[0]
print()
print("Prompt string:", repr("Question: " + first["question"] + "\n"))
print("Answer string:", repr(first["answer"] + tokenizer.eos_token))
prompt_length = sum(label == -100 for label in first_row["labels"])
answer_length = sum(label != -100 for label in first_row["labels"])
print("Token pieces: ", tokenizer.convert_ids_to_tokens(first_row["input_ids"]))
print("Prompt length:", prompt_length)
print("Answer length:", answer_length)


`Prompt length` counts labels equal to `-100`. `Answer length` counts the supervised ids. The supervised span should start at `Answer` and end at the end token.

Omitting the separate encode step and tokenizing `prompt + answer` as one string can move that cut by a token. The loss would then train on part of the question, or ignore the first piece of the answer, while the scalar loss could still fall.


## 5. Labels and masking

Three lists describe every training row. They have the same length.

| Token region | `input_ids` | `attention_mask` | `labels` |
|---|---|---|---|
| Prompt | Actual token ids | 1 | `-100` |
| Answer, including eos | Actual token ids | 1 | The same token ids |
| Padding (added in the next section) | Pad token id | 0 | `-100` |

`input_ids` is what the model reads. `attention_mask` tells attention which positions are real tokens. `labels` chooses the positions that enter the loss.

`-100` is the ignore index of `CrossEntropyLoss`. Those positions contribute neither to the numerator nor to the averaging of the loss. The prompt is still in the forward pass, so it still changes the hidden states that predict the answer. Masking removes the prompt's own next-token terms. It does not block attention to the prompt.

Hugging Face causal language models shift the labels inside the forward pass. Logits at position `t` are compared with `labels[t + 1]`. You should not shift the lists yourself. Doing both shifts scores each answer token against the wrong target.

For a row whose prompt occupies the first three positions, the scored predictions look like this:

```text
positions:   p0   p1   p2   a0   a1   eos
labels:      -100 -100 -100 a0   a1   eos

scored pairs after the internal shift:
  logit at p2   must predict a0
  logit at a0   must predict a1
  logit at a1   must predict eos
```

The prompt-to-prompt predictions are paired with `-100` and dropped. The first answer token is still supervised, because its id sits in `labels` at the answer's own index and is compared with the logit from the last prompt token.

The masked objective is ordinary next-token cross-entropy on the supervised positions:

```text
L = - (1 / sum_t m_t) * sum_t m_t * log p(z_t | z_<t)
```

`m_t` is 1 on answer and end-token positions and 0 on prompt and padding. During training, `z_<t` for an answer token includes the gold answer tokens that came before it. That training condition is called teacher forcing. At generation time the model does not get those gold tokens. It has to continue from its own previous outputs. A low training loss therefore measures imitation with the answer already on the page, which is a different test from writing the answer from the question alone.


In [ ]:
print(f"{'i':<4} {'token':<22} {'input_id':<10} {'attn':<6} {'label'}")
for index, (token_id, mask, label) in enumerate(
    zip(first_row["input_ids"], first_row["attention_mask"], first_row["labels"])
):
    token = tokenizer.convert_ids_to_tokens(token_id)
    label_text = "-100 (ignore)" if label == -100 else str(label)
    print(f"{index:<4} {token:<22} {token_id:<10} {mask:<6} {label_text}")

assert len(first_row["input_ids"]) == len(first_row["attention_mask"]) == len(first_row["labels"])
supervised = [token_id for token_id in first_row["labels"] if token_id != -100]
print()
print("Supervised text:", repr(tokenizer.decode(supervised)))


`Supervised text` should be the answer plus the end token, with no question words in it. If the question leaks into that string, the mask is wrong and the training loop will faithfully learn the wrong target.


## 6. Padding and batching

Rows have different lengths because `0 + 13` and `Answer: 21` do not tokenize to the same number of pieces as `0 + 0` and `Answer: 0`. A PyTorch batch is a rectangle, so every row in the tensor needs the same number of columns.

`pad_sequence` extends shorter rows on the right:

- `input_ids` is filled with `pad_token_id`.
- `attention_mask` is filled with `0`, so attention skips the filler.
- `labels` is filled with `-100`, so the filler is not a training target.

This notebook pads once, to the longest row in the whole training file, and then every batch reuses that rectangle. That is global padding. A larger trainer usually pads each batch only up to the longest row inside that batch. Dynamic padding wastes less compute. Here the sequences are tiny, and building the full rectangle first makes the mask easy to print. The attention mask still matters: without it, pad positions would be treated as real context. Without `-100` on padded labels, and with the pad id equal to the end id, the model would also be trained to predict an extra end token after the real one.


In [ ]:
lengths = [len(row["input_ids"]) for row in train_rows]
print("sequence lengths:", lengths)
print("shortest:", min(lengths), " longest:", max(lengths))
print("all rows already the same length:", min(lengths) == max(lengths))

for index, row in enumerate(train_rows):
    same = len(row["input_ids"]) == len(row["attention_mask"]) == len(row["labels"])
    if not same:
        raise ValueError(f"Row {index} has mismatched input_ids, attention_mask, and labels.")

input_ids = pad_sequence(
    [torch.tensor(row["input_ids"]) for row in train_rows],
    batch_first=True,
    padding_value=tokenizer.pad_token_id,
)
attention_mask = pad_sequence(
    [torch.tensor(row["attention_mask"]) for row in train_rows],
    batch_first=True,
    padding_value=0,
)
labels = pad_sequence(
    [torch.tensor(row["labels"]) for row in train_rows],
    batch_first=True,
    padding_value=-100,
)

print("input_ids:     ", tuple(input_ids.shape), input_ids.dtype)
print("attention_mask:", tuple(attention_mask.shape), attention_mask.dtype)
print("labels:        ", tuple(labels.shape), labels.dtype)


The shape is `[num_examples, padded_length]`. The first dimension is 40. The second dimension is the longest training row. `labels` uses a 64-bit integer dtype because token ids and the ignore index `-100` are integers, not class probabilities.


## 7. DataLoader

`TensorDataset` stores the three tensors and returns one row when indexed. `DataLoader` groups those rows into batches.

A batch is the set of examples that share one forward pass and one optimizer update. Shuffling the training rows changes which examples share a batch from epoch to epoch, so a single odd row does not always sit next to the same neighbors. This run does not fix the shuffle seed, so a second run can take a different path to a similar loss. The held-out loader later keeps `shuffle=False` so the evaluation order stays stable.

Batch size trades memory for the number of updates. With 40 rows and batch size 4 there are 10 updates per epoch. A batch size of 1 would use less memory and take 40 smaller steps per epoch. A batch size of 40 would take one step per epoch and would need the whole rectangle in one forward/backward pass. Four is a convenience for this toy, not a tuned value.


In [ ]:
train_dataset = TensorDataset(input_ids, attention_mask, labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"{len(train_data)} examples")
print(f"{len(train_loader)} batches per epoch at batch size {BATCH_SIZE}")
print(f"padded tensor shape: {tuple(input_ids.shape)}")
print(f"one batch shape:     ({BATCH_SIZE}, {input_ids.shape[1]})")


## 8. Load the model and optimizer

`AutoModelForCausalLM` loads a model whose forward pass predicts the next token and, when `labels` are provided, returns the masked cross-entropy as `outputs.loss`. The checkpoint already contains pretrained weights. Fine-tuning does not reinitialize them. It starts from those weights and changes them with the SFT loss.

A parameter is a tensor the optimizer is allowed to update. In this notebook every parameter is trainable. That is full fine-tuning. Methods such as LoRA freeze the base weights and train a small added update instead. They are left out on purpose so the update itself stays visible.

AdamW keeps a running average of each gradient and of each squared gradient, and it also decays the weights slightly. The learning rate scales the update. `5e-5` is the rate used for the recorded toy run. A much larger rate can move the pretrained model so far that behaviour outside this dataset falls apart. A much smaller rate can leave the weights almost unchanged after five epochs. Neither statement makes `5e-5` the right rate for a different model or dataset.

The device is where the matrix multiplies run. CUDA and MPS are faster when they exist. The model, the batch, and later the generation inputs all have to live on that same device.


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# The numbers quoted in the results sections came from a CPU run.
# Uncomment the next line to stay on CPU while revisiting that run.
# device = torch.device("cpu")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype="auto",
    local_files_only=True,
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

n_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {n_params:,}")
print("Device:", device)
print("Learning rate:", LEARNING_RATE)


## 9. The training loop

One epoch is one pass through the training loader. Each batch inside the epoch is one optimization step:

```text
batch of input_ids, attention_mask, labels
        |
        v
forward pass  ->  next-token logits, then masked cross-entropy
        |
        v
loss.backward()  ->  a gradient for each trainable parameter
        |
        v
optimizer.step()  ->  AdamW writes a small update into the pretrained weights
```

The operations in order:

1. `model.train()` turns on training behaviour. This architecture does not rely on dropout, but the call is still the standard switch before updates.
2. `optimizer.zero_grad()` clears gradients left over from the previous batch. Without it, the new gradients would be added to the old ones and the step would be larger than the loss of this batch alone.
3. The forward pass runs the model on the batch. Passing `labels` makes Hugging Face compute the shifted, masked cross-entropy.
4. `outputs.loss` is a scalar: the mean negative log probability of the supervised tokens in the batch.
5. `loss.backward()` fills `parameter.grad`. It does not change the weights. Skipping it leaves the gradients empty, and the following step has nothing new to apply.
6. `optimizer.step()` changes the weights. Skipping it computes a loss and then discards the chance to learn from it.
7. The batch losses are added up and divided by the number of batches. Every batch in this run has 4 rows, so each batch gets equal weight.
8. The printout is the average for that epoch.

Five epochs mean each of the 40 answers is seen five times, which is 50 optimizer steps. On a set this small, repeating the same targets is why the training loss can fall close to zero. The model is being asked to imitate 40 short strings, many times.


In [ ]:
epoch_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0

    for batch_input_ids, batch_attention_mask, batch_labels in train_loader:
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_mask = batch_attention_mask.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            labels=batch_labels,
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    epoch_losses.append(average_loss)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Average Loss: {average_loss:.4f}")


## 10. Training results

Read the printed loss as imitation quality under teacher forcing. A decrease means that, when the gold previous tokens are provided, the model puts more probability on the training answer tokens. It does not measure whether a freshly generated answer is correct, and it does not measure the held-out questions.

### Recorded run

These values are from one execution of this loop on the 40-row file, with AdamW at `5e-5`, batch size 4, five epochs, shuffling enabled, and the model left on CPU. They are not a general property of SFT, and a rerun can print different decimals.

| Epoch | Average training loss |
|---|---|
| 1 | 1.2441 |
| 2 | 0.0804 |
| 3 | 0.0153 |
| 4 | 0.0034 |
| 5 | 0.0023 |

The curve falls quickly because the targets are short and repeated. The next sections ask whether that fit shows up in generated answers on questions that were never updated on.


## 11. Save the fine-tuned model

The updated weights exist only in memory until they are written out. Saving stores the model configuration and tensors, plus the tokenizer, including the pad token we set. The comparison later loads this folder from disk so it is scoring the saved checkpoint rather than a variable that only exists in this kernel.

Running the next cell overwrites `models/Qwen3-0.6B-SFT` if that folder already exists.


In [ ]:
SFT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_MODEL_DIR)
tokenizer.save_pretrained(SFT_MODEL_DIR)
print("Saved fine-tuned model and tokenizer to", SFT_MODEL_DIR)


## 12. Held-out evaluation

The held-out file is prepared with the same `make_example` function, so the prompt string, the end token, and the `-100` mask match training. Its loader does not shuffle. The optimizer is not used.

`model.eval()` switches the model to evaluation behaviour. `torch.no_grad()` stops PyTorch from storing activations for a backward pass. Together they make the forward pass a measurement. Calling `optimizer.step()` here would update the weights on the evaluation questions and defeat the split.

Held-out loss uses the same teacher-forced objective as training: the gold answer is in `input_ids`, and the loss only scores those answer tokens. It can be higher than the final training loss because these pairs were not repeated for five epochs. A gap between the two losses is a hint about fit versus unseen rows. It is still not a generation score.

Generation is the other measurement. The model receives only the prompt, then appends tokens one by one from its own outputs until it emits the end token or reaches `max_new_tokens`. Greedy decoding (`do_sample=False`) always picks the most likely next token, so this check does not depend on a sampling seed. `max_new_tokens=32` is enough for `Answer: <number>` and matches the comparison below.


In [ ]:
heldout_rows = [make_example(example) for example in heldout_data]

heldout_input_ids = pad_sequence(
    [torch.tensor(row["input_ids"]) for row in heldout_rows],
    batch_first=True,
    padding_value=tokenizer.pad_token_id,
)
heldout_attention_mask = pad_sequence(
    [torch.tensor(row["attention_mask"]) for row in heldout_rows],
    batch_first=True,
    padding_value=0,
)
heldout_labels = pad_sequence(
    [torch.tensor(row["labels"]) for row in heldout_rows],
    batch_first=True,
    padding_value=-100,
)

heldout_loader = DataLoader(
    TensorDataset(heldout_input_ids, heldout_attention_mask, heldout_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model.eval()
total_loss = 0.0

with torch.no_grad():
    for batch_input_ids, batch_attention_mask, batch_labels in heldout_loader:
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_mask = batch_attention_mask.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            labels=batch_labels,
        )
        total_loss += outputs.loss.item()

heldout_loss = total_loss / len(heldout_loader)

print(f"Number of held-out examples: {len(heldout_data)}")
print(f"Training loss (final epoch): {epoch_losses[-1]:.4f}")
print(f"Held-out loss:               {heldout_loss:.4f}")


def generate_answer(model, question):
    """Continue a question prompt with greedy decoding and return only the new text."""
    prompt = f"Question: {question}\n"
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    answer_ids = output_ids[0][prompt_length:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()


print()
print("HELD-OUT GENERATION RESULTS")
for index, example in enumerate(heldout_data, start=1):
    predicted = generate_answer(model, example["question"])
    print(f"\nExample {index}")
    print("Question: ", example["question"])
    print("Expected: ", example["answer"])
    print("Predicted:", predicted)


## 13. Base model versus fine-tuned model

The fine-tuned weights are reloaded from `models/Qwen3-0.6B-SFT`. The original checkpoint is loaded again from `models/Qwen3-0.6B-Base`, which this notebook never overwrites. Both models receive the same prompt string, the same tokenizer, and the same greedy settings. Exact-match accuracy counts a prediction only when the generated string equals the `answer` field, such as `Answer: 4`. A correct sum written in any other format would count as a miss. That is deliberate: the supervised target includes the format.

The training model is deleted before the two checkpoints are loaded so the process is not holding three copies of the weights.

### What the comparison can say

It can say how often each checkpoint emits the exact target string on these 20 prompts. It cannot say that SFT taught addition in general. The base model may already solve the task. If it does, a fine-tune that only imitates similar answers can leave accuracy unchanged or make it worse, even while training loss falls.

### Recorded comparison

On this held-out file, in the CPU run described above, greedy exact-match accuracy was **100% for the base model (20/20)** and **95% for the fine-tuned model (19/20)**. Held-out loss on the fine-tuned model was **0.0361**, against a final training loss of **0.0023**.

The only mismatch was:

```text
Question:   What is 1 + 18?
Expected:   Answer: 19
Base model: Answer: 19
SFT model:  Answer: 29
```

Training contained `2 + 18` with target `Answer: 20`, and it did not contain `1 + 18`. After five epochs the fine-tuned model missed a question the base model had right. The other 19 held-out strings matched. For this toy, SFT fit the training answers and did not improve the measured skill.


In [ ]:
# Release the in-memory fine-tune before loading the two checkpoints used below.
del model
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype="auto",
    local_files_only=True,
).to(device)
base_model.eval()

sft_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    torch_dtype="auto",
    local_files_only=True,
).to(device)
sft_model.eval()

correct_base = 0
correct_sft = 0

print("BASE MODEL VS FINE-TUNED MODEL")
for index, example in enumerate(heldout_data, start=1):
    question = example["question"]
    expected = example["answer"]
    base_answer = generate_answer(base_model, question)
    sft_answer = generate_answer(sft_model, question)

    correct_base += int(base_answer == expected)
    correct_sft += int(sft_answer == expected)

    print(f"\nExample {index}")
    print("Question:  ", question)
    print("Expected:  ", expected)
    print("Base model:", base_answer)
    print("SFT model: ", sft_answer)

base_accuracy = 100.0 * correct_base / len(heldout_data)
sft_accuracy = 100.0 * correct_sft / len(heldout_data)
print()
print(f"Base model accuracy: {base_accuracy:.2f}%")
print(f"SFT model accuracy:  {sft_accuracy:.2f}%")
print(f"Absolute accuracy change: {sft_accuracy - base_accuracy:+.2f} percentage points")


## 14. Final takeaways

The mechanics in this notebook are the basic SFT loop:

```text
Data
 -> Tokenization
 -> Labels
 -> Batching
 -> Forward Pass
 -> Loss
 -> Backpropagation
 -> Optimizer Update
 -> Evaluation
```

The dataset decides which behaviour is imitated. The label mask decides which tokens are responsible for the gradient. The training loss measures that imitation while the gold answer is visible. Generation, under a frozen prompt and a frozen decoder, is the measurement of what the model produces on its own. A comparison against the untouched base checkpoint is what shows whether the update changed anything you care about.

On this addition file, the recorded update lowered exact-match accuracy by one question. That result belongs to this dataset, this checkpoint, this learning rate, and this five-epoch run. It is a useful outcome for learning SFT because it separates a falling loss from a better model.

### What this toy implementation leaves out

Real SFT pipelines often add machinery this notebook skips. Each item below changes engineering or experimental control. None of them is required to see how a masked next-token loss updates a pretrained model.

- **Larger and more diverse datasets.** Forty near-duplicate additions can be memorized. A broader instruction mixture is how people try to change general behaviour.
- **Chat templates.** Production chat models serialize roles, special tokens, and an assistant prefix with a fixed template. Training and evaluation have to use the same serializer. This notebook uses one plain string instead.
- **Dynamic padding.** Batches are padded to their own maximum length rather than the maximum of the whole dataset.
- **Validation-based checkpoint selection.** Models are saved during training, and a checkpoint is chosen by held-out behaviour rather than by the last epoch.
- **LoRA or QLoRA.** Most base weights stay frozen. A low-rank update, sometimes through a quantized base model, is trained instead of every parameter.
- **Gradient accumulation.** Several small batches are added together before `optimizer.step()`, which simulates a larger batch when memory is tight.
- **Mixed precision.** Forward and backward passes run in a lower-precision dtype to save memory and time.
- **Learning-rate scheduling.** The rate warms up or decays across steps instead of staying at one constant.
- **More rigorous evaluation.** Several prompt wordings, a fixed seed, and a task the base model does not already solve. Exact match on 20 easy sums is a teaching metric only.
